# Gestionnaires de contexte

## Implémentation d'un gestionnaire de contexte avec les méthodes magiques

Créez une classe `RedirecteurSortie` qui sera utilisable comme un gestionnaire de contexte. Cette classe stockera la valeur de `sys.stdout` à l'entrée dans le contexte, et la remplacera par l'argument qui lui est donné à sa création. À la sortie du contexte, l'ancienne sortie standard devra être restaurée. `__enter__` renverra le nouveau flux, pour qu'on puisse le récupérer avec `as`.

Avec un flux en mémoire, ce code doit afficher `Capturé : 'Bonjour\n'` :

```python
with RedirecteurSortie(io.StringIO()) as tampon:
  print("Bonjour")
print(f"Capturé : {tampon.getvalue()!r}")
```

**Documentation utile**

- [`sys.stdout`](https://docs.python.org/fr/3/library/sys.html#sys.stdout)
- [`io.StringIO`](https://docs.python.org/fr/3/library/io.html#io.StringIO)
- [Le type gestionnaire de contexte](https://docs.python.org/fr/3/library/stdtypes.html#typecontextmanager)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`__enter__` garde `sys.stdout` dans un attribut avant de le remplacer : `__exit__` en a besoin pour le restaurer. Ce que `__enter__` renvoie est ce que reçoit `as`. `__exit__` prend trois arguments, qui décrivent l'exception éventuelle, et renvoie `False` pour la laisser remonter.

```python
class RedirecteurSortie:
  def __init__(self, nouvelle_sortie):
    self.nouvelle_sortie = nouvelle_sortie

  def __enter__(self):
    self.ancienne_sortie = ...
    sys.stdout = ...
    return ...

  def __exit__(self, exc_type, exc_val, exc_tb):
    sys.stdout = ...
    return False
```

### Solution

In [ ]:
import io
import sys


class RedirecteurSortie:
  def __init__(self, nouvelle_sortie):
    self.nouvelle_sortie = nouvelle_sortie

  def __enter__(self):
    self.ancienne_sortie = sys.stdout
    sys.stdout = self.nouvelle_sortie
    return self.nouvelle_sortie

  def __exit__(self, exc_type, exc_val, exc_tb):
    sys.stdout = self.ancienne_sortie
    return False


with RedirecteurSortie(io.StringIO()) as tampon:
  print("Bonjour")
print(f"Capturé : {tampon.getvalue()!r}")

## Implémentation d'un gestionnaire de contexte avec un générateur

Recréez ce gestionnaire à l'aide de `contextlib.contextmanager`, sous la forme d'une fonction génératrice `rediriger_sortie(nouvelle_sortie)` qui cède le nouveau flux. Écrivez-la d'abord sans `try`/`finally`.

Levez ensuite une exception dans le corps du `with`, capturez-la autour du `with`, et vérifiez si `sys.stdout` est restauré, en le comparant avec `is` à sa valeur d'avant le bloc. Gardez cette valeur dans une variable, et remettez-la vous-même dans `sys.stdout` avant d'afficher le résultat : sinon, si la sortie n'est pas restaurée, plus rien ne s'affichera dans le notebook.

Sans `try`/`finally`, vous devez voir que `sys.stdout` n'est pas restauré. Ajoutez-le, et vérifiez qu'il l'est.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le code avant `yield` s'exécute à l'entrée du bloc. L'entrée reste hors du `try` : si elle échoue, il n'y a rien à restaurer. Seul le `yield` va dans le `try`, et la restauration dans le `finally`.

```python
@contextlib.contextmanager
def rediriger_sortie(nouvelle_sortie):
  ancienne_sortie = ...
  sys.stdout = ...
  try:
    yield ...
  finally:
    ...


sortie_standard = sys.stdout
try:
  with rediriger_sortie(io.StringIO()):
    raise ValueError("erreur dans le corps")
except ValueError:
  pass
restauree = ...
sys.stdout = sortie_standard
print(f"Sortie restaurée : {restauree}")
```

### Solution

In [ ]:
import contextlib
import io
import sys


@contextlib.contextmanager
def rediriger_sortie_sans_finally(nouvelle_sortie):
  ancienne_sortie = sys.stdout
  sys.stdout = nouvelle_sortie
  yield nouvelle_sortie
  sys.stdout = ancienne_sortie


@contextlib.contextmanager
def rediriger_sortie(nouvelle_sortie):
  ancienne_sortie = sys.stdout
  sys.stdout = nouvelle_sortie
  try:
    yield nouvelle_sortie
  finally:
    sys.stdout = ancienne_sortie


def verifier_restauration(gestionnaire):
  sortie_standard = sys.stdout
  try:
    with gestionnaire(io.StringIO()):
      raise ValueError("erreur dans le corps")
  except ValueError:
    pass
  restauree = sys.stdout is sortie_standard
  sys.stdout = sortie_standard
  print(f"{gestionnaire.__name__} : sortie restaurée : {restauree}")


verifier_restauration(rediriger_sortie_sans_finally)
verifier_restauration(rediriger_sortie)

En pratique, `contextlib.redirect_stdout` fait déjà tout cela.

## Implémentation d'un contexte qui supprime un type d'exception

Implémentez avec `contextlib.contextmanager` un gestionnaire de contexte `supprime(*types_exceptions)` qui supprime les exceptions des types donnés (en stoppe la propagation) dans le corps du contexte, comme `contextlib.suppress`. Une exception d'un autre type doit traverser le contexte :

```python
with supprime(NameError):
  print(abcd)  # Rien ne s'affiche, et le programme continue

with supprime(NameError, KeyError):
  {}["absente"]  # Idem

with supprime(NameError):
  1 / 0  # La ZeroDivisionError remonte
```

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une exception levée dans le corps du `with` est relancée dans le générateur, au point du `yield` : c'est là qu'on la capture. `*types_exceptions` donne un n-uplet, qu'`except` accepte tel quel. Une exception capturée et non relancée ne remonte plus.

```python
@contextlib.contextmanager
def supprime(*types_exceptions):
  try:
    ...
  except ...:
    ...
```

### Solution

In [ ]:
import contextlib


@contextlib.contextmanager
def supprime(*types_exceptions):
  try:
    yield
  except types_exceptions:
    pass


with supprime(NameError):
  print(abcd)

with supprime(NameError, KeyError):
  {}["absente"]

try:
  with supprime(NameError):
    1 / 0
except ZeroDivisionError:
  print("ZeroDivisionError n'est pas supprimée : elle traverse le contexte")

## Annulation d'une transaction

Écrivez une classe `Transaction` qui protège un dictionnaire de comptes : elle sauvegarde son contenu à l'entrée dans le contexte, et renvoie le dictionnaire. Si le corps lève une exception, `__exit__` remet le dictionnaire dans son état d'entrée et laisse l'exception remonter. Sinon, les modifications restent.

La fonction `virer` ci-dessous s'en sert. Ce code doit garder les comptes intacts au second virement, qui échoue une fois Bob déjà débité :

```python
comptes = {"alice": 100, "bob": 20}
virer(comptes, "alice", "bob", 30)
print(comptes)  # {'alice': 70, 'bob': 50}
try:
  virer(comptes, "bob", "alice", 80)
except ValueError as erreur:
  print(erreur)  # solde insuffisant sur bob
print(comptes)  # {'alice': 70, 'bob': 50}
```

**Documentation utile**

- [`dict.clear`](https://docs.python.org/fr/3/library/stdtypes.html#dict.clear)
- [`dict.update`](https://docs.python.org/fr/3/library/stdtypes.html#dict.update)

In [ ]:
def virer(comptes, source, cible, montant):
  with Transaction(comptes) as soldes:
    soldes[source] -= montant
    if soldes[source] < 0:
      raise ValueError(f"solde insuffisant sur {source}")
    soldes[cible] += montant


# Votre code ici

### Conseils supplémentaires

`__exit__` sait que le corps a levé quand `exc_type` n'est pas `None`. La restauration se fait sur place, avec `clear` et `update`, car `comptes` reste le même dictionnaire pour l'appelant : `self.comptes = self.sauvegarde` ne changerait que l'attribut. Renvoyer `False` laisse remonter l'exception, sans la relancer.

```python
class Transaction:
  def __init__(self, comptes):
    self.comptes = comptes

  def __enter__(self):
    self.sauvegarde = ...
    return ...

  def __exit__(self, exc_type, exc_val, exc_tb):
    if ...:
      self.comptes.clear()
      self.comptes.update(...)
    return ...
```

### Solution

In [ ]:
class Transaction:
  def __init__(self, comptes):
    self.comptes = comptes

  def __enter__(self):
    self.sauvegarde = dict(self.comptes)
    return self.comptes

  def __exit__(self, exc_type, exc_val, exc_tb):
    if exc_type is not None:
      self.comptes.clear()
      self.comptes.update(self.sauvegarde)
    return False


comptes = {"alice": 100, "bob": 20}
virer(comptes, "alice", "bob", 30)
print(comptes)
try:
  virer(comptes, "bob", "alice", 80)
except ValueError as erreur:
  print(erreur)
print(comptes)

## Ouverture d'une liste de fichiers (facultatif)

Écrivez une fonction `premieres_lignes(chemins)` qui ouvre tous les fichiers de la liste en même temps, avec `contextlib.ExitStack`, et renvoie la liste de leurs premières lignes, sans le saut de ligne. Enregistrez aussi avec `callback` l'affichage de `Fichiers fermés`, qui doit avoir lieu même si une ouverture échoue :

```python
print(premieres_lignes(chemins))  # Fichiers fermés, puis ['un', 'deux', 'trois']
try:
  premieres_lignes([*chemins, "absent.txt"])
except FileNotFoundError as erreur:
  print(erreur)  # Fichiers fermés, puis l'erreur
```

In [ ]:
from pathlib import Path

chemins = ["janvier.txt", "fevrier.txt", "mars.txt"]
for chemin, contenu in zip(chemins, ["un", "deux", "trois"]):
  Path(chemin).write_text(contenu + "\n", encoding="utf-8")

# Votre code ici

### Conseils supplémentaires

`pile.enter_context(gestionnaire)` entre dans le contexte et renvoie ce que renvoie son `__enter__`, ici le fichier ouvert. La pile quitte ses contextes à sa propre sortie, dans l'ordre inverse : pour s'afficher après la fermeture des fichiers, le `callback` s'enregistre avant eux.

```python
def premieres_lignes(chemins):
  with contextlib.ExitStack() as pile:
    pile.callback(...)
    fichiers = [... for chemin in chemins]
    return [... for fichier in fichiers]
```

### Solution

In [ ]:
import contextlib


def premieres_lignes(chemins):
  with contextlib.ExitStack() as pile:
    pile.callback(print, "Fichiers fermés")
    fichiers = [
      pile.enter_context(open(chemin, encoding="utf-8")) for chemin in chemins
    ]
    return [fichier.readline().strip() for fichier in fichiers]


print(premieres_lignes(chemins))
try:
  premieres_lignes([*chemins, "absent.txt"])
except FileNotFoundError as erreur:
  print(erreur)

## Réimplémentation du décorateur `contextlib.contextmanager` (difficile)

À l'aide d'un décorateur, qui créera une classe avec les méthodes `__init__`, `__enter__` et `__exit__`, réimplémentez le décorateur `contextlib.contextmanager`. Vérifiez-le en décorant `supprime` avec votre `contextmanager` : les exemples de son exercice doivent donner le même résultat.

**Documentation utile**

- [`next`](https://docs.python.org/fr/3/library/functions.html#next)
- [`generator.throw`](https://docs.python.org/fr/3/library/stdtypes.html#generator.throw)
- [`StopIteration`](https://docs.python.org/fr/3/library/exceptions.html#StopIteration)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le décorateur renvoie une classe : l'appel `supprime(NameError)` crée donc une instance, dont `__init__` reçoit les arguments et crée le générateur.

- `__enter__` appelle `next` sur le générateur, qui s'exécute jusqu'au `yield` : la valeur cédée est ce que reçoit `as`.
- Sans exception, `__exit__` appelle `next` une seconde fois : le générateur reprend après le `yield` et doit se terminer, c'est-à-dire lever `StopIteration`.
- Avec une exception, `__exit__` la relance dans le générateur, au point du `yield`, avec `throw(exc_val)`. S'il la capture et se termine, `StopIteration` est levée, et `__exit__` renvoie `True` pour la supprimer. S'il ne la capture pas, elle ressort de `throw` et remonte.
- Si le générateur cède une seconde fois, aucune `StopIteration` n'est levée : c'est une erreur, à signaler par une `RuntimeError`.

```python
def contextmanager(generateur):
  class Enveloppe:
    def __init__(self, *args, **kwargs):
      self._generateur = ...

    def __enter__(self):
      return ...

    def __exit__(self, exc_type, exc_val, exc_tb):
      try:
        if exc_type is None:
          ...
        else:
          ...
      except StopIteration:
        pass
      else:
        raise RuntimeError(...)
      return True

  return Enveloppe
```

### Solution

In [ ]:
def contextmanager(generateur):
  class Enveloppe:
    def __init__(self, *args, **kwargs):
      self._generateur = generateur(*args, **kwargs)

    def __enter__(self):
      return next(self._generateur)

    def __exit__(self, exc_type, exc_val, exc_tb):
      try:
        if exc_type is None:
          next(self._generateur)
        else:
          self._generateur.throw(exc_val)
      except StopIteration:
        pass
      else:
        raise RuntimeError("Un générateur de contexte ne doit céder qu'une fois")
      return True

  return Enveloppe


@contextmanager
def supprime(*types_exceptions):
  try:
    yield
  except types_exceptions:
    pass


with supprime(NameError):
  print(abcd)

try:
  with supprime(NameError):
    1 / 0
except ZeroDivisionError:
  print("ZeroDivisionError n'est pas supprimée : elle traverse le contexte")

Le même mécanisme en vidéo : qui exécute le code après le `yield`, et pourquoi il faut un `try`/`finally` autour : [`@contextmanager`, pas à pas](https://shuuchuu.github.io/videos/python/contextmanager-implementation.mp4).